# Classify cell density for new data

Reads new experimental data and fits a linear regression to predict cell density from protocol features.

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd
from pathlib import Path 
import os
import yaml
import torch 

In [2]:
ANNOTATION_CFG_FILE = "../../../../../inverse/loss_guidance/config/annotation/default.yaml"
with open(ANNOTATION_CFG_FILE, "r") as fb:
    annotation_cfg = yaml.safe_load(fb)
protocol_cols = annotation_cfg["protocol_columns"]
protocol_cols = [protocol for protocol in protocol_cols]

In [84]:
data_df = pd.read_csv("cell_count_protocol_df.csv", index_col=0)

In [85]:
X, y = data_df.values[:, 3:-1], data_df.values[:, -1]

In [86]:
X

array([[  0. ,   0. ,   0. , ...,   0. ,  20. ,  13. ],
       [  0. ,   0. ,   0. , ...,   0. ,  20. ,  15. ],
       [  0. ,   0. ,   0. , ...,   0. ,  20. ,  21. ],
       ...,
       [ 10. ,   2.5, 750. , ...,   0. ,  20. ,  21. ],
       [ 10. ,   2.5, 750. , ...,   0. ,  20. ,  14. ],
       [ 10. ,   2.5, 750. , ...,   0. ,  20. ,  21. ]], shape=(103, 12))

In [87]:
# Standardize X
X_scaled = np.log1p(X)

In [88]:
X_scaled

array([[0.        , 0.        , 0.        , ..., 0.        , 3.04452244,
        2.63905733],
       [0.        , 0.        , 0.        , ..., 0.        , 3.04452244,
        2.77258872],
       [0.        , 0.        , 0.        , ..., 0.        , 3.04452244,
        3.09104245],
       ...,
       [2.39789527, 1.25276297, 6.62140565, ..., 0.        , 3.04452244,
        3.09104245],
       [2.39789527, 1.25276297, 6.62140565, ..., 0.        , 3.04452244,
        2.7080502 ],
       [2.39789527, 1.25276297, 6.62140565, ..., 0.        , 3.04452244,
        3.09104245]], shape=(103, 12))

In [89]:
# Train model
model = LinearRegression()
model.fit(X_scaled, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [90]:
y_pred = model.predict(X_scaled)

In [91]:
y_pred

array([11.10845139, 11.13544335, 11.1998154 , 11.76789862, 11.84531645,
       11.38831248, 10.40786369, 10.53539871, 10.73663403, 11.12808862,
       12.21219849, 11.40320339, 11.86641099, 11.43538639, 11.898594  ,
       11.44726798, 11.91047559, 13.2451063 , 12.0617409 , 12.5249485 ,
       12.0939239 , 12.55713151, 12.07881353, 12.10580549, 12.17017755,
       13.2242585 , 13.31562251, 12.5690131 , 12.35867463, 12.45003864,
       13.50411959, 13.59548361, 12.08395463, 13.2293996 , 13.32076361,
       12.36381573, 13.50926069, 13.60062471, 13.13849604, 12.64907639,
       13.79452136, 12.17793319, 13.32337816, 12.6411408 , 13.17067905,
       13.69497143, 13.60323925, 12.66095798, 13.80640295, 12.18981478,
       13.24389573, 13.33525975, 12.65302239, 13.18256063, 13.70685302,
       12.46967587, 13.52375683, 13.61512084, 12.10359187, 13.24903683,
       13.34040085, 13.26536989, 13.71224444, 12.44887312, 13.17375555,
       13.18770173, 12.89574766, 13.6206301 , 12.38345296, 13.08

## Read new data 

In [92]:
cell_types = [
    "EryPro",
    "HSCs", 
    "preProB",
    "late_MgkPro"
    ]

In [93]:
path = Path("../../../../../project_folder/output/inverse/loss_guidance/manually_filtered/original_bloodplus/filtered/")

In [94]:
for cell_type in cell_types: 
    df = pd.read_csv(path / f"filtered_candidates_{cell_type}.csv", index_col=0)
    df_X = np.log1p(df.loc[:, protocol_cols].values)
    y_hat = model.predict(df_X)
    df["density"] = np.exp(y_hat)
    df.to_csv(path / f"filtered_candidates_{cell_type}_with_cell_count_pred.csv")